In [ ]:
import cv2
import numpy as np
import glob
import os
import re

# Function to resize images to the same height (if needed)
def resize_to_same_height(images):
    heights = [img.shape[0] for img in images]
    max_height = max(heights)
    resized_images = []
    for img in images:
        h, w = img.shape[:2]
        # Calculate new width to maintain aspect ratio
        new_width = int(w * max_height / h)
        resized = cv2.resize(img, (new_width, max_height), interpolation=cv2.INTER_AREA)
        resized_images.append(resized)
    return resized_images

# Function to extract time number from filename (e.g., "time120" -> 120)
def extract_time(filename):
    match = re.search(r'time(\d+)', filename)
    return int(match.group(1)) if match else 0

# Function to create video for a given pressure level
def create_video(pressure_level, base_dir, ref_dir, output_dir):
    # Define paths
    base_path = os.path.join(base_dir, f"0008_Feb_{pressure_level}hPa")
    ref_path = os.path.join(ref_dir, f"{pressure_level}hPa")
    output_path = os.path.join(output_dir, f"{pressure_level}hPa", f"PV_comparison_{pressure_level}hPa.mp4")

    # Ensure output directory exists
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Get lists of images and sort by time number
    nocouple_files = sorted(glob.glob(os.path.join(base_path, "*0001.png")), key=extract_time)[:120]  # First 120 files
    couple_files = sorted(glob.glob(os.path.join(base_path, "*0002.png")), key=extract_time)[:120]    # First 120 files
    ref_files = sorted(glob.glob(os.path.join(ref_path, "*.png")), key=extract_time)[:120]           # First 120 files

    # Check if we have exactly 120 files for each set
    if len(nocouple_files) != 120 or len(couple_files) != 120 or len(ref_files) != 120:
        raise ValueError(f"Expected 120 images for {pressure_level}hPa, but found: "
                         f"nocouple={len(nocouple_files)}, couple={len(couple_files)}, ref={len(ref_files)}")

    # Read the first set of images to determine output video dimensions
    img_nocouple = cv2.imread(nocouple_files[0])
    img_couple = cv2.imread(couple_files[0])
    img_ref = cv2.imread(ref_files[0])

    # Resize images to the same height
    images = resize_to_same_height([img_nocouple, img_couple, img_ref])

    # Calculate dimensions of the concatenated frame
    total_width = sum(img.shape[1] for img in images)
    height = images[0].shape[0]

    # Define video writer (1 fps, MP4 format)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_path, fourcc, 1.0, (total_width, height))

    # Loop over 120 frames
    for i in range(120):
        # Read images for the current time step
        img_nocouple = cv2.imread(nocouple_files[i])
        img_couple = cv2.imread(couple_files[i])
        img_ref = cv2.imread(ref_files[i])

        # Resize to the same height
        images = resize_to_same_height([img_nocouple, img_couple, img_ref])

        # Concatenate images horizontally
        frame = np.hstack(images)

        # Write frame to video
        video_writer.write(frame)

    # Release the video writer
    video_writer.release()
    print(f"Video for {pressure_level}hPa saved to {output_path}")

# Main execution
base_dir = "/home/weiji/restart_exam/20250307/PV_NCL"
ref_dir = "/home/weiji/restart_exam/20250221/PV_NCL/ERA52020reference"
output_dir = "/home/weiji/restart_exam/20250307/PV_NCL/ERA52020reference"

# Create videos for 10 hPa and 50 hPa
create_video(10, base_dir, ref_dir, output_dir)
create_video(50, base_dir, ref_dir, output_dir)